In [1]:
#FIXED CHUNK SIZE - CHUNKING

from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

file_path = "iowa_sales_rag_documentation_en.md"

text = Path(file_path).read_text(encoding="utf-8")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.create_documents(
    texts=[text],
    metadatas=[{"source": file_path, "chunking_strategy": "fixed_size_overlap"}],
)
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i
    chunk.metadata["chunk_length"] = len(chunk.page_content)

print(f"Chunks: {len(chunks)}")


Chunks: 20


In [8]:
# FIXED CHUNK SIZE - GET

from langchain_community.vectorstores import FAISS
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# 1. Vector store in memory
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL
)

vectorstore_fixed = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
)

# 2. Reranker
reranker = CrossEncoder(RERANKER_MODEL)


def rerank(query, docs, top_n=3):
    pairs = [
        [query, doc.page_content]
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    scored_docs = list(zip(docs, scores))
    scored_docs.sort(key=lambda x: x[1], reverse=True)

    result = []

    for doc, score in scored_docs[:top_n]:
        doc.metadata["rerank_score"] = float(score)
        result.append(doc)

    return result


# 3. Neighbor expansion on chunk_id
def expand_fixed_neighbors(docs, window=1):
    chunk_by_id = {
        chunk.metadata["chunk_id"]: chunk
        for chunk in chunks
    }

    selected_ids = set()

    for doc in docs:
        chunk_id = doc.metadata["chunk_id"]

        for neighbor_id in range(
            chunk_id - window,
            chunk_id + window + 1
        ):
            if neighbor_id in chunk_by_id:
                neighbor_chunk = chunk_by_id[neighbor_id]

                if neighbor_id != chunk_id:
                    neighbor_chunk.metadata["vector_score"] = doc.metadata.get(
                        "vector_score"
                    )
                    neighbor_chunk.metadata["rerank_score"] = doc.metadata.get(
                        "rerank_score"
                    )

                selected_ids.add(neighbor_id)

    return [
        chunk_by_id[chunk_id]
        for chunk_id in sorted(selected_ids)
    ]


# 4. Main retrieval
def retrieve_fixed_chunks(
    query,
    k=8,
    top_n=3,
    use_rerank=True,
    use_neighbor_expansion=True,
    neighbor_window=1,
    internal_print=1,
):
    # Vector search
    docs_with_scores = vectorstore_fixed.similarity_search_with_score(
        query=query,
        k=k,
    )
    if internal_print:
        print("DISTANCE STRATEGY:", vectorstore_fixed.distance_strategy)
    docs = []
    if internal_print:
        print("\nVECTOR SEARCH RESULTS")
        print("=" * 40)

    for rank, (doc, score) in enumerate(docs_with_scores, start=1):
        doc.metadata["vector_score"] = float(score)

        docs.append(doc)
        if internal_print:
            print(f"\nRANK: {rank}")
            print(f"VECTOR SCORE: {score:.4f}")
            print("-" * 40)
            print(doc.page_content[:50])

    # Rerank
    if use_rerank:
        docs = rerank(
            query=query,
            docs=docs,
            top_n=top_n,
        )
        if internal_print:
            print("\n\nRERANK RESULTS")
            print("=" * 40)

            for rank, doc in enumerate(docs, start=1):
                print(f"\nRANK: {rank}")
                print(
                    f"VECTOR SCORE: "
                    f"{doc.metadata.get('vector_score', 0):.4f}"
                )
                print(
                    f"RERANK SCORE: "
                    f"{doc.metadata.get('rerank_score', 0):.4f}"
                )
                print("-" * 40)
                print(doc.page_content[:50])
    else:
        docs = docs[:top_n]

    # Neighbor expansion
    if use_neighbor_expansion:
        docs = expand_fixed_neighbors(
            docs=docs,
            window=neighbor_window,
        )

    return docs


# TEST
results = retrieve_fixed_chunks(
    "List all the tables.", k=3
)

print("\n\nFINAL DOCUMENTS")
print("=" * 40)

for i, doc in enumerate(results, start=1):
    print(f"\nDOCUMENT {i}")
    print(
        f"VECTOR SCORE: "
        f"{doc.metadata.get('vector_score', 'N/A')}"
    )
    print(
        f"RERANK SCORE: "
        f"{doc.metadata.get('rerank_score', 'N/A')}"
    )
    print("-" * 40)
    print(doc.page_content[0:50])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

DISTANCE STRATEGY: DistanceStrategy.EUCLIDEAN_DISTANCE

VECTOR SEARCH RESULTS

RANK: 1
VECTOR SCORE: 1.1878
----------------------------------------
## General model description

The `Iowa Sales` mod

RANK: 2
VECTOR SCORE: 1.1942
----------------------------------------
## Notes for RAG systems

This document is designe

RANK: 3
VECTOR SCORE: 1.2518
----------------------------------------
## Fact table iowa_sales

The `iowa_sales` table i


RERANK RESULTS

RANK: 1
VECTOR SCORE: 1.2518
RERANK SCORE: -2.1696
----------------------------------------
## Fact table iowa_sales

The `iowa_sales` table i

RANK: 2
VECTOR SCORE: 1.1942
RERANK SCORE: -3.0655
----------------------------------------
## Notes for RAG systems

This document is designe

RANK: 3
VECTOR SCORE: 1.1878
RERANK SCORE: -5.0414
----------------------------------------
## General model description

The `Iowa Sales` mod


FINAL DOCUMENTS

DOCUMENT 1
VECTOR SCORE: 1.1877543926239014
RERANK SCORE: -5.041425704956055
-----------

In [3]:
#SECTION CHUNK SIZE - CHUNKING
from pathlib import Path
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

file_path = "iowa_sales_rag_documentation_en.md"

text = Path(file_path).read_text(encoding="utf-8")

# 1. Split po sekcjach Markdown
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3"),
    ],
    strip_headers=False,
)

section_docs = markdown_splitter.split_text(text)

# DODAJ METADATA TUTAJ
for section_id, section_doc in enumerate(section_docs):
    section_doc.metadata["section_id"] = section_id
    section_doc.metadata["source"] = file_path
    section_doc.metadata["file_name"] = Path(file_path).name
    section_doc.metadata["original_section_length"] = len(section_doc.page_content)

# 2. Fallback split
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

final_chunks = recursive_splitter.split_documents(section_docs)

# Dodatkowe metadata dla final chunks
for chunk_id, chunk in enumerate(final_chunks):
    chunk.metadata["chunk_id"] = chunk_id
    chunk.metadata["chunk_length"] = len(chunk.page_content)

print(f"Section docs: {len(section_docs)}")
print(f"Final chunks: {len(final_chunks)}")

Section docs: 15
Final chunks: 21


In [4]:
# SECTION CHUNK - GET

from langchain_community.vectorstores import FAISS
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

# 1. Embedding local model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 2. In-memory vector store
vectorstore_section = FAISS.from_documents(
    documents=final_chunks,
    embedding=embeddings,
)

RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANKER_MODEL)


def rerank(query, docs, top_n=3):
    pairs = [
        [query, doc.page_content]
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    scored_docs = list(zip(docs, scores))
    scored_docs.sort(key=lambda x: x[1], reverse=True)

    return [
        doc
        for doc, score in scored_docs[:top_n]
    ]


def expand_section_neighbors(docs, window=1):
    chunk_by_id = {
        chunk.metadata["chunk_id"]: chunk
        for chunk in final_chunks
    }

    selected_ids = set()

    for doc in docs:
        chunk_id = doc.metadata["chunk_id"]
        section_id = doc.metadata["section_id"]

        for neighbor_id in range(chunk_id - window, chunk_id + window + 1):
            neighbor = chunk_by_id.get(neighbor_id)

            if neighbor is None:
                continue

            if neighbor.metadata["section_id"] == section_id:
                selected_ids.add(neighbor_id)

    return [
        chunk_by_id[chunk_id]
        for chunk_id in sorted(selected_ids)
    ]


def print_doc_with_score(doc, score):
    print("\n" + "=" * 40)
    print("VECTOR SCORE:", score)
    print("METADATA:", str(doc.metadata)[:50])
    print("-" * 40)
    print(doc.page_content[:50])


def retrieve_section_chunks(
    query,
    k=8,
    top_n=3,
    use_rerank=True,
    use_neighbor_expansion=True,
    neighbor_window=1,
    internal_print=1,
):
    # 1. Vector search
    docs_with_scores = vectorstore_section.similarity_search_with_score(
        query=query,
        k=k,
    )

    if internal_print:
        print("DISTANCE STRATEGY:", vectorstore_section.distance_strategy)

    docs = []

    for doc, score in docs_with_scores:
        doc.metadata["vector_score"] = float(score)
        docs.append(doc)

        if internal_print:
            # Print score together with document
            print_doc_with_score(doc, score)

    # 2. Rerank
    if use_rerank:
        docs = rerank(
            query=query,
            docs=docs,
            top_n=top_n,
        )
    else:
        docs = docs[:top_n]

    # 3. Neighbor expansion
    if use_neighbor_expansion:
        docs = expand_section_neighbors(
            docs=docs,
            window=neighbor_window,
        )

    return docs


retrieved_docs = retrieve_section_chunks(
    "List all the tables.",
    k=3,
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

DISTANCE STRATEGY: DistanceStrategy.EUCLIDEAN_DISTANCE

VECTOR SCORE: 1.1942145
METADATA: {'h1': 'Iowa Sales Data Documentation for RAG', 'h
----------------------------------------
## Notes for RAG systems  
This document is design

VECTOR SCORE: 1.2077059
METADATA: {'h1': 'Iowa Sales Data Documentation for RAG', 'h
----------------------------------------
## General model description  
The `Iowa Sales` mo

VECTOR SCORE: 1.2616818
METADATA: {'h1': 'Iowa Sales Data Documentation for RAG', 'h
----------------------------------------
## Fact table iowa_sales  
The `iowa_sales` table 


In [ ]:
#LLM CLIENT FORM FIXED AND SECTION CHUNKING METHOD
from openai import OpenAI


client = OpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="lm-studio",
)


def build_context(docs):
    context_parts = []

    for i, doc in enumerate(docs, start=1):
        context_parts.append(
            f"""
            DOCUMENT: {i}
            METADATA: {doc.metadata}
            CONTENT: {doc.page_content}"""
)

    return "\n\n".join(context_parts)


def ask_llm(question, docs):
    context = build_context(docs)

    prompt = f"""
    You are a helpful RAG assistant. Answer the question ONLY using the provided context.
    If the answer is not in the context, say that the information is missing.
    QUESTION: {question}
    CONTEXT: {context}"""

    response = client.chat.completions.create(
        model="lmstudio-community/qwen3-8b-mlx",
        messages=[
            {
                "role": "system",
                "content": "You answer questions using retrieved context.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        temperature=0,
    )

    usage = response.usage

    return {
        "answer": response.choices[0].message.content,
        "input_tokens": getattr(usage, "prompt_tokens", None),
        "output_tokens": getattr(usage, "completion_tokens", None),
        "total_tokens": getattr(usage, "total_tokens", None),
        "usage": usage.model_dump() if hasattr(usage, "model_dump") else usage,
    }

In [9]:
#FIXED AND SECTION CHUNK SIZE - SIDE TO SIDE

query = """
I want information about relationships between iowa tables. Additionally column types in related columns and description of these columns.
"""
docs_fixed = retrieve_fixed_chunks(
    query=query,
    k=8,
    top_n=3,
    use_rerank=True,
    use_neighbor_expansion=True,
    neighbor_window=1,
    internal_print=0,
)

docs_section = retrieve_section_chunks(
    query=query,
    k=8,
    top_n=3,
    use_rerank=True,
    use_neighbor_expansion=True,
    internal_print=0,
)



answer_fixed = ask_llm(
    question=query,
    docs=docs_fixed,
)

answer_section = ask_llm(
    question=query,
    docs=docs_section,
)


def print_answer(label, result):
    print(f"---{label}----")
    print(f"Input tokens: {result['input_tokens']}")
    print(f"Output tokens: {result['output_tokens']}")
    print(f"Total tokens: {result['total_tokens']}")
    print(result["answer"])


print_answer("FIXED", answer_fixed)
print_answer("SECTION", answer_section)

---FIXED----
Input tokens: 2659
Output tokens: 1226
Total tokens: 3885


The relationships between Iowa tables involve foreign keys in the `iowa_sales` fact table linking to dimension tables. Below is the information on relationships, column types, and descriptions:

---

### **1. `iowa_sales` to `iowa_category`**
- **Join Condition**:  
  `iowa_sales.CategoryID = iowa_category.CategoryID`  
- **Columns**:
  - `iowa_sales.CategoryID` (INT): Foreign key to `iowa_category.CategoryID` (BIGINT).  
  - `iowa_category.CategoryID` (BIGINT): Primary key for the category dimension.  
  - `iowa_category.CategoryName` (STRING): Stores category names like "Whiskey" or "Vodka".  

---

### **2. `iowa_sales` to `iowa_date`**
- **Join Condition**:  
  `iowa_sales.DateID = iowa_date.DateID`  
- **Columns**:
  - `iowa_sales.DateID` (INT): Foreign key to `iowa_date.DateID` (INT).  
  - `iowa_date.DateID` (INT): Primary key for the date dimension.  
  - `iowa_date.CalendarDate` (DATE): Full calendar date